# EPUB Audiobook - Qwen3-TTS Test

Test the Qwen3-TTS VoiceDesign model on Google Colab with GPU acceleration.

**Requirements:** Colab runtime with GPU (T4 or better)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShayC0derBui/audiobook/blob/main/notebooks/test_tts_colab.ipynb)

In [ ]:
# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected. Go to Runtime > Change runtime type > GPU")

In [ ]:
# Install epub-audiobook from GitHub
!pip install -q "git+https://github.com/ShayC0derBui/audiobook.git#egg=epub-audiobook[colab]"

In [ ]:
# Verify installation
!epub-audiobook --help

In [ ]:
# Test TTS synthesis with default voice
!epub-audiobook test-tts \
    --text "Hello, this is a test of the Qwen3 text to speech system running on Google Colab." \
    --output /content/test_output.wav \
    --profile colab

In [ ]:
# Play the generated audio
from IPython.display import Audio
Audio('/content/test_output.wav')

In [ ]:
# Test with custom voice prompt
!epub-audiobook test-tts \
    --text "The old wizard spoke slowly, his voice deep and resonant, echoing through the ancient halls." \
    --voice-prompt "A deep, mature male voice with British accent, slow and deliberate pace, warm baritone" \
    --output /content/test_voice_design.wav \
    --profile colab

In [ ]:
# Play custom voice
from IPython.display import Audio
Audio('/content/test_voice_design.wav')

## Full EPUB Conversion

Upload an EPUB file and convert it to audiobook.

In [ ]:
# Upload EPUB file
from google.colab import files
uploaded = files.upload()
epub_filename = list(uploaded.keys())[0]
print(f"Uploaded: {epub_filename}")

In [ ]:
# Inspect EPUB structure
!epub-audiobook inspect "$epub_filename"

In [ ]:
# Convert EPUB to audiobook (adjust language and voice as needed)
!epub-audiobook convert "$epub_filename" \
    --output /content/audiobook_output \
    --profile colab \
    --language en \
    --voice-prompt "A clear, pleasant narrator voice, moderate pace, neutral accent"

In [ ]:
# Download results as ZIP
import shutil
shutil.make_archive('/content/audiobook', 'zip', '/content/audiobook_output')
files.download('/content/audiobook.zip')

## Direct Python API Usage

Use the TTS client directly for more control.

In [ ]:
# Direct API usage
from epub_audiobook.config import AppConfig, RuntimeProfile, TTSConfig
from epub_audiobook.tts.qwen_client import QwenTTSClient
from pathlib import Path
import soundfile as sf

# Create config for Colab
config = AppConfig(
    profile=RuntimeProfile.COLAB,
    input_path=Path("."),
)

# Initialize client
client = QwenTTSClient(config)

# Synthesize with custom voice
tts_config = TTSConfig(
    voice_design_prompt="Young female narrator, clear and expressive, medium pace",
    language="en",
)

audio = client.synthesize(
    text="Once upon a time, in a land far away, there lived a curious young girl.",
    tts_config=tts_config,
)

sr = client.sample_rate
sf.write('/content/direct_api_test.wav', audio, sr)
print(f"Generated {len(audio)/sr:.1f}s of audio at {sr}Hz")

from IPython.display import Audio
Audio('/content/direct_api_test.wav')